# RT-DETR Zero-Shot on T4 — removing the "CPU¹" footnote

Every RT-DETR number in this project so far (`rtdetr_v2_r18_zeroshot`,
`rtdetr_ultralytics_l_zeroshot` in `results/eval/comparison.csv`) was
measured locally on CPU -- never on the same T4 hardware used for every
other method's headline numbers. This was `kalan_isler.md`'s top-priority
open item. This notebook re-runs both RT-DETR checkpoints on a T4, zero
training involved -- it only changes the hardware, not the model or
weights.

**Bug fixed alongside this (2026-09-16):** `src/methods/rtdetr/model.py`'s
HF-based `RtDetrV2Detector` never moved its model off CPU -- there was no
`.to(device)` call anywhere in it. Running the original script unmodified
on this T4 would have silently stayed on CPU and just reproduced the same
CPU numbers again under a different label. Added a `device` parameter
(default `"cpu"`, so the already-recorded local CPU result is unaffected
and still reproducible); this notebook passes `device: cuda`. The
Ultralytics-based `RtDetrUltralyticsDetector` needed no such fix --
Ultralytics models auto-select CUDA when available, the same mechanism
that already produced `yolo26n_zeroshot_T4`'s verified GPU numbers.

Writes to **new** result directories (`results/rtdetr_zeroshot_T4/`,
`results/rtdetr_ultralytics_zeroshot_T4/`) -- the original CPU-run
predictions are left untouched, so CPU and T4 stay directly comparable
afterward (same pattern as `yolo26n_zeroshot` vs. `yolo26n_zeroshot_T4`,
which showed the hardware barely moves mAP: 0.6935 vs. 0.6933).

Atomic like `03_gpu_method_comparison.ipynb` / `04_sam3_zeroshot_eval.ipynb`
-- one step per cell, calling the same `scripts/*.py` used everywhere
else, no notebook-only logic (`CLAUDE.md` Section 9).

**Before running:** upload `test_frames.zip` to
`/content/drive/MyDrive/object-detection/test_frames.zip` if it isn't
there already (same file the other GPU notebooks use). No Hugging Face
login needed this time -- `PekingU/rtdetr_v2_r18vd` is a public checkpoint,
unlike SAM3.


## Step 0 — Setup: clone the repo, install dependencies, confirm GPU

In [ ]:
import os
if not os.path.exists("object-detection-drone"):
    !git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!git pull origin main --no-edit --no-rebase
!pip install -q -r requirements.txt


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (do not proceed -- this run only has a point if it's actually on a GPU)")


## Step 1 — Get the gold test set's frames

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

TEST_FRAMES_ZIP = "/content/drive/MyDrive/object-detection/test_frames.zip"
!unzip -q -o "{TEST_FRAMES_ZIP}" -d .
!echo "test frames: $(find data/processed/frames/test -name '*.jpg' | wc -l)"


## Step 2 — RT-DETRv2-R18 (Hugging Face) on the T4

Same checkpoint, same `conf=0.25`, same native 640x640 resolution as the
original CPU run (`configs/28_rtdetr_zeroshot.yaml`) -- only `device` and
`output_dir` are patched, in a session-local config copy so the committed
config (and its recorded CPU result) stays untouched.

In [ ]:
import yaml

config_path = "configs/28_rtdetr_zeroshot.yaml"
config = yaml.safe_load(open(config_path))
config["device"] = "cuda"
config["output_dir"] = "results/rtdetr_zeroshot_T4"
yaml.safe_dump(config, open("configs/28_T4.yaml", "w"), sort_keys=False)
print(open("configs/28_T4.yaml").read())


In [ ]:
!python scripts/28_rtdetr_zeroshot_eval.py --config configs/28_T4.yaml


## Step 3 — RT-DETR-l (Ultralytics) on the T4

No `device` override needed -- Ultralytics auto-selects CUDA when
available, the same mechanism already verified by `yolo26n_zeroshot_T4`.
Only `output_dir` is patched. The `rtdetr-l.pt` checkpoint auto-downloads
if it isn't already present on this VM.

In [ ]:
import yaml

config_path = "configs/30_rtdetr_ultralytics_zeroshot.yaml"
config = yaml.safe_load(open(config_path))
config["output_dir"] = "results/rtdetr_ultralytics_zeroshot_T4"
yaml.safe_dump(config, open("configs/30_T4.yaml", "w"), sort_keys=False)
print(open("configs/30_T4.yaml").read())


In [ ]:
!python scripts/30_rtdetr_ultralytics_zeroshot_eval.py --config configs/30_T4.yaml


## Step 4 — Evaluate both against the gold test set

Same harness as every other method (`scripts/06_evaluate.py`) -- mAP,
per-video breakdown, precision/recall/F1 at conf=0.25, appended to
`results/eval/comparison.csv` under new `_T4`-suffixed method names.

In [ ]:
!python scripts/06_evaluate.py \
    --predictions results/rtdetr_zeroshot_T4/predictions.json \
    --method-name rtdetr_v2_r18_zeroshot_T4 \
    --config configs/06_evaluate.yaml

!python scripts/06_evaluate.py \
    --predictions results/rtdetr_ultralytics_zeroshot_T4/predictions.json \
    --method-name rtdetr_ultralytics_l_zeroshot_T4 \
    --config configs/06_evaluate.yaml


## Step 5 — Sanity check: T4 vs. the original CPU run

mAP/precision/recall should barely move (same weights, same thresholds,
only the hardware differs) -- if any of these jump noticeably, something
about the patched config is wrong, not just "GPU vs CPU noise". FPS is
the actual point of this run.

In [ ]:
import pandas as pd

df = pd.read_csv("results/eval/comparison.csv")
display(df[df["method"].isin([
    "rtdetr_v2_r18_zeroshot", "rtdetr_v2_r18_zeroshot_T4",
    "rtdetr_ultralytics_l_zeroshot", "rtdetr_ultralytics_l_zeroshot_T4",
])])


## Step 6 — FPS from the manifests

In [ ]:
import json, glob

for prefix, label in [
    ("28_rtdetr_zeroshot_eval", "RT-DETRv2-R18"),
    ("30_rtdetr_ultralytics_zeroshot_eval", "RT-DETR-l"),
]:
    manifest_path = sorted(glob.glob(f"results/manifests/{prefix}_*.json"))[-1]
    m = json.load(open(manifest_path))
    fps = m["num_frames"] / m["elapsed_seconds"]
    print(f"{label} (T4): {m['num_frames']} frames / {m['elapsed_seconds']}s -> {fps:.1f} FPS   ({manifest_path})")


## Step 7 — Push results back to GitHub

Text artifacts only -- no review images, no checkpoints (matches every
other GPU notebook's convention; review-image dumps are gitignored
anyway).

In [ ]:
import getpass
gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
!git add results/eval results/manifests results/rtdetr_zeroshot_T4/predictions.json results/rtdetr_ultralytics_zeroshot_T4/predictions.json
!git commit -m "Add RT-DETR zero-shot T4 benchmark (both checkpoints, removes the CPU-only caveat)"
!git pull origin main --no-edit --no-rebase
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main
